In [ ]:
# This was originally the example of obstacle avoidance from lecture.
# The repository is https://github.com/TobiaMarcucci/olrc-code/blob/main/examples/chapter4/obstacle_avoidance_dcf.ipynb
# I just made some tweaks.

import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
# Some circle obstacles each of radius 1 are created here.
radius = .1
centers = np.array([
    [.7, .3],
    [.8, .8],
    [.3, .2],
    [.3, .7],
    [.6, .6],
])

# Initial and final point.
x0 = np.array([0, 0])
xK = np.array([1, 1])
K = 20  #This is how many points are on the trajectory.
# Variables.
X = [cp.Variable((5, 5), PSD=True) for k in range(K)]

# Objective function.
obj = 0
P = np.zeros((5, 5))    # This is doing the QP relaxation
P[1:, 1:] = np.eye(4)
P[1:3, 3:] = - np.eye(2)
P[3:, 1:3] = - np.eye(2)
for Xk in X:
    obj += cp.trace(P @ Xk)

# Initial and final conditions.
constraints = [
    X[0][0, 1:3] == x0,
    X[0][1:3, 1:3] == cp.outer(x0, x0),
    X[0][1:3, 3:] == cp.outer(x0, X[0][0, 3:]),
    X[-1][0, 3:] == xK,
    X[-1][3:, 3:] == cp.outer(xK, xK),
    X[-1][1:3, 3:] == cp.outer(X[-1][0, 1:3], xK)]

# Coherence constraints.
for Xk in X:
    constraints.append(Xk[0, 0] == 1)
for Xk, Xkp in zip(X[:-1], X[1:]):
    constraints.append(Xk[0, 3:] == Xkp[0, 1:3])
    constraints.append(Xk[3:, 3:] == Xkp[1:3, 1:3])

# Obstacle avoidance constraints.
for Xk in X[1:]:
    for center in centers:
        P = np.zeros((5, 5))
        P[0, 0] = radius ** 2 - center.dot(center)
        P[0, 1:3] = center
        P[1:3, 0] = center
        P[1:3, 1:3] = - np.eye(2)
        constraints.append(cp.trace(P @ Xk) <= 0)

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(solver='MOSEK')
# Exctract optimal solution.
points = np.array([Xk[0, 1:3].value for Xk in X] + [X[-1][0, 3:].value])

# Plot result.
plt.figure()
plt.axis('equal')
for c in centers:
    patch = Circle(c, radius, facecolor='lightcoral', edgecolor='black')
    plt.gca().add_patch(patch)
plt.plot(*points.T, marker='o', label=f"Relaxation value = {prob.value}")
plt.legend()
plt.show()